In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.metrics import precision_score, recall_score, f1_score

# calc_metrics computes the metrics (MAE, MAPE, RMSE) for energy, macronutrients, mass predictions
# Note: MAPE is computed as defined by Nutrition5k
# (see: https://github.com/google-research-datasets/Nutrition5k/blob/main/scripts/compute_eval_statistics.py)
def calc_metrics(predictions, ground_truth):
    # Mean Absolute Error (MAE)
    mae = mean_absolute_error(ground_truth, predictions)

    # Mean Absolute Percentage Error (MAPE)
    #mape = mean_absolute_percentage_error(ground_truth, predictions)
    mape = 100 * mae / np.mean(ground_truth)

    # Root Mean Square Error (RMSE)
    rmse = mean_squared_error(ground_truth, predictions, squared=False)

    return rmse, mae, mape

# calc_class_metrics instead computes the metrics for ingredient predictions
def calc_class_metrics(predictions, ground_truth):

    p = precision_score(ground_truth, predictions, average="micro", zero_division=0.0)
    r = recall_score(ground_truth, predictions, average="micro")
    f1 = f1_score(ground_truth, predictions, average="micro")
    return p, r, f1
    
# create_dict_metrics structures a dictionary to comprise metrics for all the predictions.
# It is mostly used for logging/debugging purposes. Actual, final predictions used in the paper are obtained by analyzing
# the test-time predictions (later computed based on the numpy files with the final predictions).
# This function is called for training, validation, and testing purposes, so that the logging (within wandb) contains many info.
def create_dict_metrics(phase, predictions, ground_truth, ingredient_predictions, ingredient_groundtruth):
    step = ['Calories', 'Fat', 'Carbs', 'Proteins', 'Mass']
    loss = mean_squared_error(ground_truth, predictions)
    info_dict = {f'{phase}/Loss': loss.item()}
    # For each of the 5 components, we add a small epsilon (0.0000000001) to avoid 0-divisions.
    for i in range(5):
        rounded_gt = 1e-10 + ground_truth[:, i] #.cpu().detach().numpy()
        rmse, mae, mape = calc_metrics(predictions[:, i], #.cpu().detach().numpy(), 
                                        rounded_gt)  # to avoid 0 errors
    
        info_dict[f'{phase}/{step[i]}/RMSE'] = rmse
        info_dict[f'{phase}/{step[i]}/MAE'] = mae
        info_dict[f'{phase}/{step[i]}/MAPE'] = mape

    # Ingredient predictions arrive here as logits (output of sigmoid function). These are transformed into
    # binary predictions (ingredient is there or not) based on a threshold, which we define as 0.5.
    p, r, f1 = calc_class_metrics(ingredient_predictions>0.5, ingredient_groundtruth>0.5)
    info_dict[f'{phase}/Ingredients/P'] = p
    info_dict[f'{phase}/Ingredients/R'] = r
    info_dict[f'{phase}/Ingredients/F1'] = f1

    return info_dict

### Setting up the paths

In [ ]:
def set_paths(setup):
    if setup == "ita_orig":
        meta_dish_cafe1_path = 'metadata/ita_orig/Clean_DB1_ITAorig_FINALE_REP3.xlsx'
        meta_dish_cafe2_path = 'metadata/ita_orig/Clean_DB2_ITAorig_FINALE_REP3.xlsx'
        meta_path = 'metadata/ita_orig/Clean_DB1_itawr.xlsx'  # unused
        meta_recipes_cafe1_path = 'metadata/ita_orig/Clean_DB1_ITAorig_totali_FINALE.xlsx'
        meta_recipes_cafe2_path = 'metadata/ita_orig/Clean_DB2_ITAorig_totali_FINALE.xlsx'
    elif setup in ["ita_corr", "ita_corr_filt1"]:
        meta_dish_cafe1_path = 'metadata/ita_corr/Clean_DB1_ITAcorr_FINALE_REP3.xlsx'
        meta_dish_cafe2_path = 'metadata/ita_corr/Clean_DB2_ITAcorr_FINALE_REP3.xlsx'
        meta_path = 'metadata/ita_corr/metadata_Clean_DBs.xlsx'
        meta_recipes_cafe1_path = 'metadata/ita_corr/CleanDB1_totali.xlsx'
        meta_recipes_cafe2_path = 'metadata/ita_corr/CleanDB2_totali.xlsx'
    elif setup == "usa_orig":
        meta_dish_cafe1_path = 'metadata/usa_orig/dish_metadata_cafe1.csv'
        meta_dish_cafe2_path = 'metadata/usa_orig/dish_metadata_cafe2.csv'
        meta_path = 'metadata/usa_orig/ingredients_metadata.csv'
        meta_recipes_cafe1_path = 'metadata/usa_orig/dish_metadata_cafe1.csv'
        meta_recipes_cafe2_path = 'metadata/usa_orig/dish_metadata_cafe2.csv'
    elif setup in ["usa_corr", "usa_corr_filt1"]:
        meta_dish_cafe1_path = 'metadata/usa_corr/dish_metadata_cafe1_USAcorr_FINALE_REP3.xlsx'
        meta_dish_cafe2_path = 'metadata/usa_corr/dish_metadata_cafe2_USAcorr_FINALE_REP3.xlsx'
        meta_path = 'metadata/usa_corr/metadata_ingredients_Rachi.xlsx'
        meta_recipes_cafe1_path = 'metadata/usa_corr/dish_metadata_cafe1.xlsx'
        meta_recipes_cafe2_path = 'metadata/usa_corr/dish_metadata_cafe2.xlsx'
    else:
        assert False, "choose a setup in ['ita_corr_old', 'ita_corr', 'ita_orig', 'usa_orig', 'usa_corr']"

    return meta_dish_cafe1_path, meta_dish_cafe2_path, meta_path, meta_recipes_cafe1_path, meta_recipes_cafe2_path

In [ ]:
import csv 
# parse_csv_file is used for loading the metadata in the original Nutrition5k database.
# It is not really used for this paper, but it's useful for reproducing older results.
# (see Bianco et al, "2D Prediction of the Nutritional Composition of Dishes from Food Images:
# Deep Learning Algorithm Selection and Data Curation Beyond the Nutrition5k Project" Nutrients 2025).
def parse_csv_file(file_path):
    data = {}
    
    with open(file_path, 'r', newline='') as csv_file:
        csv_reader = csv.reader(csv_file)
        # The CSV files contain one recipe per row. First, the row contains data for the recipe (ID, total calories, etc).
        # Then, for each recorded ingredient, it contains similar fields (in groups of 7 columns) at the ingredient level.
        # Everything is stored in a dictionary, which becomes the output of this function.
        for row in csv_reader:
            dish_info = {
                "Dish ID": row[0],
                "Total Calories": float(row[1]),
                "Total Mass": float(row[2]),
                "Total Fat": float(row[3]),
                "Total Carbohydrates": float(row[4]),
                "Total Protein": float(row[5]),
                "Ingredients": []
            }
            
            # Extract the ingredients in groups of 7 columns
            ingredient_groups = [row[i:i+7] for i in range(6, len(row), 7)]
            
            for ingredient_group in ingredient_groups:
                ingr_dict = {
                    "Ingredient ID": ingredient_group[0],
                    "Ingredient Name": ingredient_group[1].lower(),
                    "Ingredient Grams": float(ingredient_group[2]),
                    "Ingredient Calories": float(ingredient_group[3]),
                    "Ingredient Fat": float(ingredient_group[4]),
                    "Ingredient Carbohydrates": float(ingredient_group[5]),
                    "Ingredient Protein": float(ingredient_group[6])
                }
                dish_info["Ingredients"].append(ingr_dict)
            
            data[row[0]] = dish_info
    
    return data

In [ ]:
# parse_xlsx_file is the loading function for the ITA metadata files.
# They are Excel files, and have some differences in the name and setup of columns compared to the USA counterpart.
# Per-recipe information is available at recipe_data_file_path, whereas file_path points at per-ingredient(-per-recipe) information.
# Per-recipe information has one row per recipe, without ingredient-level information.
# Per-ingredient, on the other hand, has N rows per recipe, with N=number of ingredients in the recipe.
def parse_xlsx_file(file_path, return_df_recipe_macro=True, recipe_data_file_path=None):
    _verbose = False
    data = pd.read_excel(file_path, header=1)
    data.drop(0, inplace=True)  # duplicate row with Italian header
    data.rename(columns={"Unnamed: 0": "DISH ID", "Unnamed: 1": "INGREDIENT ID", "Unnamed: 2": "NAME", "Unnamed: 3": "total mass"}, inplace=True)

    # We drop the first block of metadata because they contain nutritional information per 100g.
    # We are interested in the second block of metadata, containing per-portion nutrition metadata.
    data.drop([data.columns[i] for i in range(4, 92, 1)], axis=1, inplace=True)  # drop the "per 100g" columns
    if _verbose:
        print("="*20)
        print(data.head())
    data.ffill(inplace=True)  # fill empty recipe_id with last valid (for ingredients)
    data.rename(columns={cn: cn[:-2] for cn in data.columns if cn.endswith(".1")}, inplace=True)  # remove the extra ".1" added for duplicate names
    if _verbose:
        print(data.head())
        try:
            print(data.groupby("DISH ID").get_group("dish_1561662216"))
        except:
            print(data.groupby("DISH ID").get_group("dish_1573762622"))

    if return_df_recipe_macro:

        # the following block of code normalizes the column names
        if file_path != recipe_data_file_path:
            df_recipes = pd.read_excel(recipe_data_file_path)
            cols = {c: c.replace("_", " ").lower() for c in df_recipes.columns if c.lower() not in ["dish_id"]}
            df_recipes.rename(columns=cols, inplace=True)
            df_recipes.rename(columns={
                'Energy, Rec with fibre (kcal)'.lower(): 'total calories',
                'Energy(kcal)'.lower(): 'total calories',
                'Available carbohydrates (MSE)'.lower(): 'total carbohydrates',
                'Total fat'.lower(): 'total fat',
                'Total protein'.lower(): 'total protein',
                'total_mass'.lower(): 'total mass'},
                inplace=True)  # to obtain a similar header as the N5K
    
            cols = {c: c.replace("_", " ").lower() for c in data.columns if c.lower() not in ["dish_id"]}
            data.rename(columns=cols, inplace=True)
            data.rename(columns={
                'Energy, Rec with fibre (kcal)'.lower(): 'ingredient calories',
                'Energy(kcal)'.lower(): 'ingredient calories',
                'Available carbohydrates (MSE)'.lower(): 'ingredient carbohydrates',
                'Total fat'.lower(): 'ingredient fat',
                'Total protein'.lower(): 'ingredient protein',
                'total mass'.lower(): 'ingredient mass'},
                inplace=True)  # to obtain a similar header as the N5K
    
        else:
            df_recipes = pd.read_excel(recipe_data_file_path)
            # print(list(df_recipes.columns))
            cols = {c: c.replace("_", " ").lower() for c in df_recipes.columns if c.lower() not in ["dish_id"]}
            df_recipes.rename(columns=cols, inplace=True)
            df_recipes.rename(columns={
                'dish_ID': 'DISH_ID',
                'DISH_id': 'DISH_ID',
                'energy': 'total calories',
                'energy (kcal)': 'total calories',
                'carb': 'total carbohydrates',
                'fat': 'total fat',
                'protein': 'total protein',
                # 'prot': 'total protein',
                'mass': 'total mass'},
                inplace=True)  # to obtain a similar header as the N5K

            cols = {c: c.replace("_", " ").lower() for c in data.columns if c.lower() not in ["dish_id"]}
            data.rename(columns=cols, inplace=True)
            data.rename(columns={
                'dish_ID': 'DISH_ID',
                'DISH_id': 'DISH_ID',
                'energy': 'ingredient calories',
                'energy (kcal)': 'ingredient calories',
                'carb': 'ingredient carbohydrates',
                'fat': 'ingredient fat',
                'protein': 'ingredient protein',
                # 'prot': 'ingredient protein',
                'mass': 'ingredient mass'},
                inplace=True)  # to obtain a similar header as the N5K

        # Create a dictionary with per-recipe information
        dict_df = {di: dd for di, dd in zip(df_recipes.DISH_ID, df_recipes.to_dict(orient="records"))}
        if _verbose:
            try:
                print(dict_df["dish_1561662216"])
            except:
                print(dict_df["dish_1573762622"])

        if _verbose:
            print(data.head())
        tmp = {}
        # We update dict_df to also include ingredient-level information, leveraging side effects and pointers in python.
        for k, v in dict_df.items():
            new_v = v
            new_v["Ingredients"] = [{
                    "Ingredient ID": ing["dish id"],
                    "Ingredient Name": ing["name"].lower(),
                    "Ingredient Grams": float(ing["ingredient mass"]),
                    "Ingredient Calories": float(ing["ingredient calories"]),
                    "Ingredient Fat": float(ing["ingredient fat"]),
                    "Ingredient Carbohydrates": float(ing["ingredient carbohydrates"]),
                    "Ingredient Protein": float(ing["ingredient protein"])
            } for ing_ix, ing in data.groupby("dish id").get_group(k).iterrows()]
            tmp[k] = new_v

        if _verbose:
            try:
                print(tmp["dish_1561662216"])
            except:
                print(tmp["dish_1573762622"])

        return dict_df
    
    return data

# parse_usa_corr_file is a second loading function, dedicated for the USA corrected setup.
# It is similar to parse_csv_file, with only minor changes due to the different file format (Excel).
def parse_usa_corr_file(file_path, return_df_recipe_macro=True, recipe_data_file_path=None):
    _verbose = False
    data = pd.read_excel(file_path)
    if _verbose:
        print(data)
    dict_df = {}
    for row_ix, row in data.iterrows():
        dish_info = {
            "Dish ID": row[0],
            "Total Calories": float(row[1]),
            "Total Mass": float(row[2]),
            "Total Fat": float(row[3]),
            "Total Carbohydrates": float(row[4]),
            "Total Protein": float(row[5]),
            "Ingredients": []
        }
        
        # Extract the ingredients in groups of 7 columns
        ingredient_groups = [row[i:i+7] for i in range(6, len(row), 7)]
        
        for ingredient_group in ingredient_groups:
            if not any(ingredient_group.isna()):
                ingr_dict = {
                    "Ingredient ID": ingredient_group[0],
                    "Ingredient Name": ingredient_group[1].lower(),
                    "Ingredient Grams": float(ingredient_group[2]),
                    "Ingredient Calories": float(ingredient_group[3]),
                    "Ingredient Fat": float(ingredient_group[4]),
                    "Ingredient Carbohydrates": float(ingredient_group[5]),
                    "Ingredient Protein": float(ingredient_group[6])
                }
                dish_info["Ingredients"].append(ingr_dict)
        
        dict_df[row[0]] = dish_info
    # print(dict_df["dish_1565972591"])
    # print(dict_df["dish_1566245398"])
    if _verbose:
        print(dict_df[list(dict_df.keys())[0]])

    return dict_df

In [ ]:
train_names = open('../nutrition5k_dataset/dish_ids/splits/rgb_train_ids.txt', 'r').readlines()
train_names = [tn.strip() for tn in train_names]
# val_names = np.random.choice(range(len(train_names)), size=int(len(train_names) * 0.1), replace=False)
# val_names = [train_names[i] for i in val_names]
# train_names = [tn for tn in train_names if tn not in val_names]

test_names = open('../nutrition5k_dataset/dish_ids/splits/rgb_test_ids.txt', 'r').readlines()
test_names = [tn.strip() for tn in test_names]
print(f"split info: {len(train_names)}, {len(test_names)}")



### Creating the CSV files for further analyses

In [ ]:
import os

# We get the list of recipe IDs in the correct order, 
# by loading the metadata and filtering it based on the test split.
def get_correct_order_recipes(setup):
    meta_dish_cafe1_path, meta_dish_cafe2_path, meta_path, meta_recipes_cafe1_path, meta_recipes_cafe2_path = set_paths(setup)
    recipe_ids = []

    if "xlsx" in meta_dish_cafe1_path:
        if "ita" in meta_dish_cafe1_path.lower():
            meta_dish_cafe1 = parse_xlsx_file(meta_dish_cafe1_path, True, meta_recipes_cafe1_path)
            # meta_dish_cafe2 = parse_xlsx_file(meta_dish_cafe2_path, True, meta_recipes_cafe2_path)
        
        else:
            meta_dish_cafe1 = parse_usa_corr_file(meta_dish_cafe1_path, True, meta_recipes_cafe1_path)
            # meta_dish_cafe2 = parse_usa_corr_file(meta_dish_cafe2_path, True, meta_recipes_cafe2_path)
        try:
            meta_ingredients = pd.read_excel(meta_path)
        except:
            meta_ingredients = pd.read_csv(meta_path)
    else:
        meta_dish_cafe1 = parse_csv_file(meta_dish_cafe1_path)
        meta_dish_cafe2 = parse_csv_file(meta_dish_cafe2_path)
        meta_ingredients = pd.read_csv(meta_path)
    
    for dish_id, dish_dict in meta_dish_cafe1.items():
        if dish_id in test_names:
            recipe_ids.append(dish_id)

    cc = 0
    filtered_recipe_ids = []
    for dish_id in recipe_ids:
        frames_path = "nutrition5k_dataset/imagery/side_angles"
        num_frames = len(os.listdir(os.path.join(frames_path, dish_id, "frames_sampled5")))
        if num_frames < 2:
            # print(cc, num_frames, os.path.join(frames_path, dish_id, "frames_sampled5"))
            cc += 1
        else:
            filtered_recipe_ids.append(dish_id)
    print(filtered_recipe_ids[0], len(recipe_ids), "->", len(filtered_recipe_ids))
    return filtered_recipe_ids


In [ ]:
# We define a function for structuring the predictions and ground truth values in a dictionary,
# which is then saved as a row of a dataframe to be saved as CSV.
def add_row(data_dict, preds, gts, INGR_PRED, INGR_GTS, INGR_MAP, recipe_id, model_name):
    data_dict.append({**{
        'recipe_id': recipe_id,
        'model': model_name,
        'cal_pred': preds[0],
        'fat_pred': preds[1],
        'carb_pred': preds[2],
        'prot_pred': preds[3],
        'mass_pred': preds[4],
        'cal_gt': gts[0],
        'fat_gt': gts[1],
        'carb_gt': gts[2],
        'prot_gt': gts[3],
        'mass_gt': gts[4],
    },
    **{f"{ingname.replace(',', '')}_pred": INGR_PRED[ingid] for ingname, ingid in INGR_MAP.items() if ingname not in ["", "deprecated"]},
    **{f"{ingname.replace(',', '')}_gt": (1 if INGR_GTS[ingid] > 0.5 else 0) for ingname, ingid in INGR_MAP.items() if ingname not in ["", "deprecated"]},
    })

    return data_dict

def save_dict(data_dict, save_path):
    pd.DataFrame(data_dict).to_csv(save_path, index=False)

# We create a folder for saving the CSVs to be created.
os.makedirs("df_preds_INGREDIENTS", exist_ok=True)

In [ ]:
# We define a numerically stable sigmoid function, which is used for transforming the ingredient predictions (logits) into probabilities.

# numerically stable sigmoid: https://stackoverflow.com/questions/51976461/optimal-way-of-defining-a-numerically-stable-sigmoid-function-for-a-list-in-pyth
def _positive_sigmoid(x):
    return 1 / (1 + np.exp(-x))


def _negative_sigmoid(x):
    # Cache exp so you won't have to calculate it twice
    exp = np.exp(x)
    return exp / (exp + 1)


def sigmoid(x):
    positive = x >= 0
    # Boolean array inversion is faster than another comparison
    negative = ~positive

    # empty contains junk hence will be faster to allocate
    # Zeros has to zero-out the array after allocation, no need for that
    # See comment to the answer when it comes to dtype
    result = np.empty_like(x, dtype=float)
    result[positive] = _positive_sigmoid(x[positive])
    result[negative] = _negative_sigmoid(x[negative])

    return result

In [1]:
import json

In [ ]:
# here we precompute the list of recipe IDs in the correct order for the test split, for both setups.
correct_order_recipes = {}
for setup in ["usa_corr", "ita_corr", "usa_corr_filt1", "ita_corr_filt1"]:
    correct_order_recipes[setup] = get_correct_order_recipes(setup)

In [ ]:
def calc_stats_ASL(ARCH, SETUP, opt_pfx=""):
    filtered_recipe_ids = correct_order_recipes[SETUP]
    mlp_type = "_deepMLP" if ARCH == "2+2" else ""

    ingmap = json.load(open(f"map_ingredients2class_run4_{SETUP}.json"))
    for arch in ["R101_IN1k", "ViT-B-16_IN1k", "IncV3"]:
        data_dict = []
        metrics_dicts = []
        a100_is_used = "A100_" if arch == "IncV3" or "filt1" in SETUP else ""
        #pfx_e = f"PREDICTIONS/_env__{a100_is_used}"
        pfx_e = ""
        for run_ix in range(5):
            preds_gts = np.load(f"{pfx_e}predictions_TrainWithIngredients_ASL_gammaN5.0P0.01.0_microTstOnly_{arch}_run{run_ix}_MLPperTask{mlp_type}_TestEns_{SETUP}.npy")
            preds, gts = preds_gts[:, :5], preds_gts[:, 5:]
            ingpr_ingts = np.load(f"{pfx_e}predictions_Ingredients_ASL_gammaN5.0P0.01.0_microTstOnly_{arch}_run{run_ix}_MLPperTask{mlp_type}_TestEns_{SETUP}.npy")
            ingprs, inggts = ingpr_ingts[:, :ingpr_ingts.shape[1]//2], ingpr_ingts[:, ingpr_ingts.shape[1]//2:]
    
            for cn, recipe_id in enumerate(filtered_recipe_ids):
                data_dict = add_row(data_dict, preds[cn], gts[cn], sigmoid(ingprs[cn]), inggts[cn], ingmap, recipe_id, f"{arch}_{ARCH}")
            metrics_dict = create_dict_metrics("Testing", preds, gts, sigmoid(ingprs), inggts)
            metrics_dicts.append(metrics_dict)
    
        save_dict(data_dict, f"df_preds_INGREDIENTS/df_{arch}_{ARCH}_{SETUP}_ASL_gammaN5P0_1_MEAN.csv")
    
        print(arch)
        summary = ""
        for metric in ["Calories", "Mass", "Fat", "Carbs", "Proteins"]:
            mae_vals = np.array([m[f"Testing/{metric}/MAE"] for m in metrics_dicts])
            mape_vals = np.array([m[f"Testing/{metric}/MAPE"] for m in metrics_dicts])
            rmse_vals = np.array([m[f"Testing/{metric}/RMSE"] for m in metrics_dicts])
            summary += f"{np.mean(rmse_vals)}, {np.std(rmse_vals)}, {np.mean(mae_vals)}, {np.std(mae_vals)}, {np.mean(mape_vals)},".replace(",", ";").replace(".", ",")
        for cm in ["P", "R", "F1"]:
            f1_vals = np.array([m[f"Testing/Ingredients/{cm}"] for m in metrics_dicts])
            print(f"{cm}: {np.mean(f1_vals):.4f}")
        print(summary)
        print()

In [ ]:
for s in ["usa_corr_filt1", "ita_corr_filt1", "usa_corr", "ita_corr"]:
    for a in ["2+2"]:
        print("="*10, s, a)
        calc_stats_ASL(a, s)

In [ ]:
def calc_stats_ASL_MEDIAN(ARCH, SETUP, opt_pfx=""):
    filtered_recipe_ids = correct_order_recipes[SETUP]
    mlp_type = "_deepMLP" if ARCH == "2+2" else ""

    ingmap = json.load(open(f"map_ingredients2class_run4_{SETUP}.json"))
    for arch in ["R101_IN1k", "ViT-B-16_IN1k", "IncV3"]:
        data_dict = []
        metrics_dicts = []
        a100_is_used = "A100_" if arch == "IncV3" or "filt1" in SETUP else ""
        #pfx_e = f"PREDICTIONS/_env__{a100_is_used}"
        pfx_e = ""
        for run_ix in range(5):
            preds_gts = np.load(f"{pfx_e}predictions_MEDIAN_TrainWithIngredients_ASL_gammaN5.0P0.01.0_microTstOnly_{arch}_run{run_ix}_MLPperTask{mlp_type}_TestEns_{SETUP}.npy")
            preds, gts = preds_gts[:, :5], preds_gts[:, 5:]
            ingpr_ingts = np.load(f"{pfx_e}predictions_MEDIAN_Ingredients_ASL_gammaN5.0P0.01.0_microTstOnly_{arch}_run{run_ix}_MLPperTask{mlp_type}_TestEns_{SETUP}.npy")
            ingprs, inggts = ingpr_ingts[:, :ingpr_ingts.shape[1]//2], ingpr_ingts[:, ingpr_ingts.shape[1]//2:]
    
            for cn, recipe_id in enumerate(filtered_recipe_ids):
                data_dict = add_row(data_dict, preds[cn], gts[cn], sigmoid(ingprs[cn]), inggts[cn], ingmap, recipe_id, f"{arch}_{ARCH}")
            metrics_dict = create_dict_metrics("Testing", preds, gts, sigmoid(ingprs), inggts)
            metrics_dicts.append(metrics_dict)
    
        save_dict(data_dict, f"df_preds_INGREDIENTS/df_{arch}_{ARCH}_{SETUP}_ASL_gammaN5P0_1.csv")
    
        print(arch)
        summary = ""
        for metric in ["Calories", "Mass", "Fat", "Carbs", "Proteins"]:
            mae_vals = np.array([m[f"Testing/{metric}/MAE"] for m in metrics_dicts])
            mape_vals = np.array([m[f"Testing/{metric}/MAPE"] for m in metrics_dicts])
            rmse_vals = np.array([m[f"Testing/{metric}/RMSE"] for m in metrics_dicts])
            summary += f"{np.mean(rmse_vals)}, {np.std(rmse_vals)}, {np.mean(mae_vals)}, {np.std(mae_vals)}, {np.mean(mape_vals)},".replace(",", ";").replace(".", ",")
        for cm in ["P", "R", "F1"]:
            f1_vals = np.array([m[f"Testing/Ingredients/{cm}"] for m in metrics_dicts])
            print(f"{cm}: {np.mean(f1_vals):.4f}")
        print(summary)
        print()

In [ ]:
for s in ["usa_corr_filt1", "ita_corr_filt1", "usa_corr", "ita_corr"]:
    for a in ["2+2"]:
        print("="*10, s, a)
        calc_stats_ASL_MEDIAN(a, s)